# 00 · Setup de Unity Catalog — ElectroCasa

Crea el catálogo del ambiente, los schemas por capa, el volume de landing, las funciones de máscara y los permisos de los tres grupos.
Se ejecuta **una vez por catálogo** (`electrocasa_dev` y `electrocasa_prod`) **antes** de desplegar el bundle.

**Requisitos previos (desde la interfaz, no con API/CLI/SDK):**
1. Settings > Identity and access > Groups: crear `electrocasa_ingenieria`, `electrocasa_analistas`, `electrocasa_auditoria`.
2. Agregarte como miembro de `electrocasa_ingenieria` (el dueño del pipeline debe estar en ese grupo para que las tablas derivadas de Empleados guarden valores reales).

In [0]:
dbutils.widgets.text("catalogo", "electrocasa_dev", "Catálogo del ambiente")
CAT = dbutils.widgets.get("catalogo")

G_ING = "electrocasa_ingenieria"
G_ANA = "electrocasa_analistas"
G_AUD = "electrocasa_auditoria"
FUENTES = ["ventas", "catalogo", "empleados", "resenas", "devoluciones", "sucursales"]
print("Catálogo:", CAT)

## 1. Catálogo, schemas por capa y volume de landing

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CAT} COMMENT 'Proyecto integrador ElectroCasa'")
for capa in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.{capa} COMMENT 'Capa {capa} del modelo Medallion'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CAT}.default.vol_landing COMMENT 'Landing de archivos por fuente'")

for fuente in FUENTES:
    dbutils.fs.mkdirs(f"/Volumes/{CAT}/default/vol_landing/{fuente}/")

display(spark.sql(f"SHOW SCHEMAS IN {CAT}"))
display(dbutils.fs.ls(f"/Volumes/{CAT}/default/vol_landing/"))

## 2. Funciones de máscara (dni y salario)

- `dni`: los miembros de `electrocasa_ingenieria` ven el valor real; el resto ve solo los últimos 3 dígitos (`*****678`).
- `salario`: los miembros de `electrocasa_ingenieria` ven el valor real; el resto ve `NULL` (la máscara debe devolver el mismo tipo `DECIMAL(12,2)` de la columna, por eso no puede ser `***`).

La máscara se declara en el schema de las tablas del pipeline (cláusula `MASK`), no con un `ALTER TABLE` posterior.

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CAT}.silver.mask_dni(dni STRING)
RETURNS STRING
COMMENT 'Enmascara el DNI salvo para electrocasa_ingenieria'
RETURN CASE
  WHEN is_account_group_member('{G_ING}') THEN dni
  WHEN dni IS NULL THEN NULL
  ELSE concat(repeat('*', greatest(length(dni) - 3, 0)), right(dni, 3))
END
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CAT}.silver.mask_salario(salario DECIMAL(12,2))
RETURNS DECIMAL(12,2)
COMMENT 'Oculta el salario salvo para electrocasa_ingenieria'
RETURN CASE WHEN is_account_group_member('{G_ING}') THEN salario ELSE NULL END
""")

# Variante si en Free Edition no se pudieron crear los grupos (documentar en el README):
#   RETURN CASE WHEN current_user() = '<tu correo>' THEN dni ELSE ... END
spark.sql(f"USE CATALOG {CAT}")
display(spark.sql("SHOW USER FUNCTIONS IN silver"))

## 3. Permisos por grupo

| Grupo | Acceso |
|---|---|
| `electrocasa_ingenieria` | Lectura y escritura sobre bronze, silver y gold (+ volume de landing) |
| `electrocasa_analistas` | Solo lectura sobre gold |
| `electrocasa_auditoria` | Solo lectura sobre gold + evidencia de auditoría y lineage |

Los permisos a nivel de schema se heredan a todas las tablas actuales y futuras.

In [0]:
grants = [
    # Ingeniería: lectura y escritura en las tres capas
    f"GRANT USE CATALOG ON CATALOG {CAT} TO `{G_ING}`",
    *[f"GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE, CREATE MATERIALIZED VIEW, CREATE FUNCTION, EXECUTE "
      f"ON SCHEMA {CAT}.{capa} TO `{G_ING}`" for capa in ["bronze", "silver", "gold"]],
    f"GRANT USE SCHEMA ON SCHEMA {CAT}.default TO `{G_ING}`",
    f"GRANT READ VOLUME, WRITE VOLUME ON VOLUME {CAT}.default.vol_landing TO `{G_ING}`",
    # Analistas: solo lectura sobre gold
    f"GRANT USE CATALOG ON CATALOG {CAT} TO `{G_ANA}`",
    f"GRANT USE SCHEMA, SELECT ON SCHEMA {CAT}.gold TO `{G_ANA}`",
    # Auditoría: solo lectura sobre gold
    f"GRANT USE CATALOG ON CATALOG {CAT} TO `{G_AUD}`",
    f"GRANT USE SCHEMA, SELECT ON SCHEMA {CAT}.gold TO `{G_AUD}`",
]
for g in grants:
    spark.sql(g)
    print("OK ", g)

In [0]:
for grupo in [G_ING, G_ANA, G_AUD]:
    print(f"===== {grupo}")
    display(spark.sql(f"SHOW GRANTS `{grupo}` ON CATALOG {CAT}"))
    for capa in ["bronze", "silver", "gold"]:
        display(spark.sql(f"SHOW GRANTS `{grupo}` ON SCHEMA {CAT}.{capa}"))

## 4. Demostración de `REVOKE`

Se otorga a `electrocasa_analistas` un privilegio que **no** le corresponde (`USE SCHEMA` + `SELECT` sobre silver), se revoca y se muestra `SHOW GRANTS` antes, durante y después. **Captura las tres salidas para el README.**

In [0]:
print("1) ANTES (debe estar vacío)")
display(spark.sql(f"SHOW GRANTS `{G_ANA}` ON SCHEMA {CAT}.silver"))

In [0]:
spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {CAT}.silver TO `{G_ANA}`")
print("2) DESPUÉS DEL GRANT INDEBIDO")
display(spark.sql(f"SHOW GRANTS `{G_ANA}` ON SCHEMA {CAT}.silver"))

In [0]:
spark.sql(f"REVOKE USE SCHEMA, SELECT ON SCHEMA {CAT}.silver FROM `{G_ANA}`")
print("3) DESPUÉS DEL REVOKE (vacío otra vez)")
display(spark.sql(f"SHOW GRANTS `{G_ANA}` ON SCHEMA {CAT}.silver"))

## 6. Verificación de la membresía (puede tardar ~1 minuto en reconocerse)

In [0]:
display(spark.sql(f"""
SELECT current_user() AS usuario,
       is_account_group_member('{G_ING}') AS en_ingenieria,
       is_account_group_member('{G_ANA}') AS en_analistas,
       is_account_group_member('{G_AUD}') AS en_auditoria,
       {CAT}.silver.mask_dni('45678912') AS prueba_mask_dni
"""))

In [0]:
G_AUD = "electrocasa_auditoria"

grants_sistema = [
    f"GRANT USE CATALOG ON CATALOG system TO `{G_AUD}`",
    f"GRANT USE SCHEMA ON SCHEMA system.access TO `{G_AUD}`",
    f"GRANT SELECT ON TABLE system.access.audit TO `{G_AUD}`",
    f"GRANT SELECT ON TABLE system.access.table_lineage TO `{G_AUD}`",
    f"GRANT SELECT ON TABLE system.access.column_lineage TO `{G_AUD}`",
]
for g in grants_sistema:
    try:
        spark.sql(g)
        print("OK    ", g)
    except Exception as e:
        print("ERROR ", g)
        print("      ", str(e).splitlines()[0][:400])

In [0]:
%sql
DESCRIBE HISTORY electrocasa_dev.bronze.tracking_envios;

In [0]:
%sql
SELECT count(*) AS eventos FROM electrocasa_dev.bronze.tracking_envios;

In [0]:
%sql
SELECT metodo_pago_original, count(*) AS n
FROM electrocasa_dev.silver.ventas
WHERE metodo_pago = 'otro'
GROUP BY 1 ORDER BY 2 DESC